In [1]:
!pip install -q pymupdf

from pathlib import Path
from google.colab import drive

# Keep all preprocessing paths and thresholds in one place.
CONFIG = {
    "project_dir": Path("/content/drive/MyDrive/doculens_lava"),
    "output_dir": Path("/content/drive/MyDrive/doculens_lava/outputs"),
    "data_dir": Path("/content/lava_data"),
    "train_pdf_dir": Path("/content/lava_data/train_pdfs/train_pdfs"),
    "test_pdf_dir": Path("/content/lava_data/test_pdfs/test_pdfs"),
    "min_text_chars": 50,
    "ocr_dpi": 200,
    "enable_ocr": True,
    "enable_vlm": True,
    "enable_refusal_gate": False,
}


def setup_preprocess_workspace(config):
    """
    Input: CONFIG dictionary with Drive and local data paths.
    Output: Mounts Drive, creates output folder, and prints PDF counts found on disk.
    """
    drive.mount("/content/drive")
    config["output_dir"].mkdir(parents=True, exist_ok=True)

    train_count = len(list(config["train_pdf_dir"].glob("*.pdf"))) if config["train_pdf_dir"].exists() else 0
    test_count = len(list(config["test_pdf_dir"].glob("*.pdf"))) if config["test_pdf_dir"].exists() else 0

    print("Output folder ready:", config["output_dir"])
    print("Train PDFs found:", train_count, "| Test PDFs found:", test_count)

    # Warn early if this new notebook runtime does not have the dataset downloaded yet.
    if train_count == 0 or test_count == 0:
        print("ACTION NEEDED: Run the Kaggle download cell next before extracting pages.")


setup_preprocess_workspace(CONFIG)

Mounted at /content/drive
Output folder ready: /content/drive/MyDrive/doculens_lava/outputs
Train PDFs found: 0 | Test PDFs found: 0
ACTION NEEDED: Run the Kaggle download cell next before extracting pages.


In [2]:
import os
import shutil
import subprocess
import zipfile
from pathlib import Path


def setup_kaggle_key(project_dir):
    """
    Input: Path to your Google Drive project folder containing kaggle.json.
    Output: Copies kaggle.json into ~/.kaggle with private permissions.
    """
    source_key = project_dir / "kaggle.json"
    kaggle_folder = Path.home() / ".kaggle"
    target_key = kaggle_folder / "kaggle.json"

    if not source_key.exists():
        raise FileNotFoundError("kaggle.json not found in MyDrive/doculens_lava/.")

    # Kaggle CLI requires the key inside ~/.kaggle with 600 permissions.
    kaggle_folder.mkdir(exist_ok=True)
    shutil.copy(source_key, target_key)
    os.chmod(target_key, 0o600)


def download_and_unzip_data(competition_name, data_dir):
    """
    Input: Kaggle competition slug and local destination folder.
    Output: Downloads and extracts the dataset if not already present.
    """
    data_dir.mkdir(parents=True, exist_ok=True)
    zip_path = data_dir / f"{competition_name}.zip"

    # Skip re-downloading if the CSV is already extracted in this session.
    if not (data_dir / "train.csv").exists():
        subprocess.run(
            ["kaggle", "competitions", "download", "-c", competition_name, "-p", str(data_dir)],
            check=True,
        )
        with zipfile.ZipFile(zip_path, "r") as archive:
            archive.extractall(data_dir)
        zip_path.unlink()  # Remove zip right away to keep Colab disk free.


setup_kaggle_key(CONFIG["project_dir"])
download_and_unzip_data("lava-challenge-2026", CONFIG["data_dir"])

train_pdfs = sorted(CONFIG["train_pdf_dir"].glob("*.pdf"))
test_pdfs = sorted(CONFIG["test_pdf_dir"].glob("*.pdf"))
print("Ready -> Train PDFs:", len(train_pdfs), "| Test PDFs:", len(test_pdfs))

Ready -> Train PDFs: 5 | Test PDFs: 200


In [3]:
import pymupdf
import pandas as pd


def get_language_from_id(file_id):
    """
    Input: Document file_id string (for example, 'j_0030' or 'v_0104').
    Output: Language code ('ja', 'vi', or 'unknown').
    """
    # Filenames start with 'j_' for Japanese and 'v_' for Vietnamese.
    if file_id.startswith("j_"):
        return "ja"
    if file_id.startswith("v_"):
        return "vi"
    return "unknown"


def extract_single_pdf(pdf_path, split_name, min_chars):
    """
    Input: Path to one PDF, split label ('train' or 'test'), and min character threshold.
    Output: List of dictionaries, one dictionary per page.
    """
    document = pymupdf.open(pdf_path)
    file_id = pdf_path.stem
    language = get_language_from_id(file_id)
    page_rows = []

    for page_index in range(len(document)):
        page = document[page_index]
        page_text = page.get_text("text").strip()
        char_count = len(page_text)
        image_count = len(page.get_images())
        # Only flag for OCR if text is missing AND an image is actually present.
        needs_ocr = (char_count < min_chars) and (image_count > 0)

        page_rows.append({
            "file_id": file_id,
            "split": split_name,
            "language": language,
            "page_number": page_index + 1,  # 1-indexed to match train.csv.
            "text": page_text,
            "char_count": char_count,
            "image_count": image_count,
            "needs_ocr": needs_ocr,
            "ocr_done": False,
        })

    document.close()
    return page_rows


def extract_all_pdfs(train_pdfs, test_pdfs, min_chars):
    """
    Input: Lists of train PDF paths, test PDF paths, and min character threshold.
    Output: Combined pandas DataFrame containing every page from all PDFs.
    """
    all_rows = []

    for pdf_path in train_pdfs:
        all_rows.extend(extract_single_pdf(pdf_path, "train", min_chars))

    for pdf_path in test_pdfs:
        all_rows.extend(extract_single_pdf(pdf_path, "test", min_chars))

    return pd.DataFrame(all_rows)


pages_table = extract_all_pdfs(train_pdfs, test_pdfs, CONFIG["min_text_chars"])

print("Total pages extracted:", len(pages_table))
print("Total pages needing OCR:", int(pages_table["needs_ocr"].sum()))
print("\nOCR workload by split and language:")
print(pages_table.groupby(["split", "language"])["needs_ocr"].sum())
print("\nFirst 3 rows:")
print(pages_table[["file_id", "split", "language"]].head(3))

MuPDF error: format error: cmsOpenProfileFromMem failed

MuPDF error: format error: No default Layer config

Total pages extracted: 4772
Total pages needing OCR: 868

OCR workload by split and language:
split  language
test   ja          791
       vi           77
train  ja            0
       vi            0
Name: needs_ocr, dtype: int64

First 3 rows:
  file_id  split language
0  j_0030  train       ja
1  j_0030  train       ja
2  j_0030  train       ja


Cell 4: Test EasyOCR on 2 flagged pages first

In [4]:
!pip install -q easyocr

import time
import numpy as np
from PIL import Image
import easyocr


def render_page_image(pdf_path, page_number, dpi):
    """
    Input: Path to a PDF, 1-based page number, and render DPI.
    Output: NumPy RGB image array of that page.
    """
    document = pymupdf.open(pdf_path)
    page = document[page_number - 1]  # PyMuPDF uses 0-based indexing.
    pixmap = page.get_pixmap(dpi=dpi, colorspace=pymupdf.csRGB)
    document.close()

    # Convert the raw PDF bitmap into an RGB image array that EasyOCR accepts.
    image = Image.frombytes("RGB", [pixmap.width, pixmap.height], pixmap.samples)
    return np.array(image)


def extract_ocr_text(reader, pdf_path, page_number, dpi):
    """
    Input: EasyOCR reader, PDF path, 1-based page number, and DPI.
    Output: Extracted text as a single string.
    """
    image_array = render_page_image(pdf_path, page_number, dpi)
    # detail=0 returns only text strings; paragraph=True groups nearby lines cleanly.
    lines = reader.readtext(image_array, detail=0, paragraph=True)
    return "\n".join(lines).strip()


def test_one_ocr_page(pages_table, pdf_dir, language, reader, dpi):
    """
    Input: pages_table, folder of PDFs, language code, EasyOCR reader, and DPI.
    Output: Prints timing and a text preview for the first flagged page of that language.
    """
    flagged = pages_table[(pages_table["needs_ocr"] == True) & (pages_table["language"] == language)]
    sample_row = flagged.iloc[0]
    pdf_path = pdf_dir / f"{sample_row['file_id']}.pdf"

    start_time = time.time()
    ocr_text = extract_ocr_text(reader, pdf_path, sample_row["page_number"], dpi)
    elapsed = time.time() - start_time

    print(f"=== OCR TEST ({language}) | {sample_row['file_id']} page {sample_row['page_number']} ===")
    print(f"Time: {elapsed:.2f}s | Extracted chars: {len(ocr_text)}")
    print("Preview:", ocr_text[:200].replace("\n", " "))


# EasyOCR requires separate readers for Japanese and Vietnamese.
ja_reader = easyocr.Reader(["ja", "en"], gpu=True)
vi_reader = easyocr.Reader(["vi", "en"], gpu=True)

test_one_ocr_page(pages_table, CONFIG["test_pdf_dir"], "ja", ja_reader, CONFIG["ocr_dpi"])
test_one_ocr_page(pages_table, CONFIG["test_pdf_dir"], "vi", vi_reader, CONFIG["ocr_dpi"])

Progress: |██████████████████████████████████████████████████| 100.0% Complete

Progress: |██████████████████████████████████████████████████| 100.0% Complete

Progress: |██████████████████████████████████████████████████| 100.0% Complete

AssertionError: Invalid item number: i=np.int64(35).

In [5]:
def render_page_image(pdf_path, page_number, dpi):
    """
    Input: Path to a PDF, 1-based page number, and render DPI.
    Output: NumPy RGB image array of that page.
    """
    document = pymupdf.open(pdf_path)
    # Convert numpy int64 from pandas into a plain Python int so PyMuPDF accepts it.
    page = document[int(page_number) - 1]
    pixmap = page.get_pixmap(dpi=dpi, colorspace=pymupdf.csRGB)
    document.close()

    image = Image.frombytes("RGB", [pixmap.width, pixmap.height], pixmap.samples)
    return np.array(image)


# Reuse the already-loaded GPU readers so we do not re-download models.
test_one_ocr_page(pages_table, CONFIG["test_pdf_dir"], "ja", ja_reader, CONFIG["ocr_dpi"])
test_one_ocr_page(pages_table, CONFIG["test_pdf_dir"], "vi", vi_reader, CONFIG["ocr_dpi"])

=== OCR TEST (ja) | j_0001 page 36 ===
Time: 5.16s | Extracted chars: 118
Preview: Duplication fortimeinterval total 昌 103  102 ち 菖 101 109 109 102 1o1 Time difference (per 5ooOs)  5 重複ベクトルの時間間隔 36 104
=== OCR TEST (vi) | v_0002 page 1 ===
Time: 4.18s | Extracted chars: 1543
Preview: TẬP ĐOÀN XĂNG DẦU VIỆT NAM CỘNG HÒA XÃ HỘI CHỦ NGHĨA VIỆT NAM Độc Lập Tự Do Hạnh Phúc Hà Nội ngày 26 tháng 04 năm 2024 Số: 01/2024/PLX-BB-ĐHĐCĐ PETROLIMEX BIÊN BẢN HỌP ĐẠI HỘI ĐỒNG CỔ ĐÔNG THƯỜNG NIÊN


In [6]:
import time

# Add checkpoint settings to CONFIG (copy these two lines to Cell 1 later).
CONFIG["pages_parquet_path"] = CONFIG["output_dir"] / "pages.parquet"
CONFIG["checkpoint_every"] = 50


def load_or_create_checkpoint(pages_table, save_path):
    """
    Input: Fresh pages_table DataFrame and Path to pages.parquet on Drive.
    Output: Existing checkpoint DataFrame if found, otherwise saves and returns pages_table.
    """
    # Resume from Drive if an earlier run already finished some OCR pages.
    if save_path.exists():
        print("Resuming from existing checkpoint:", save_path)
        return pd.read_parquet(save_path)

    pages_table.to_parquet(save_path, index=False)
    print("Created new checkpoint:", save_path)
    return pages_table


def ocr_single_row(row, config, ja_reader, vi_reader):
    """
    Input: One row from pages_table, CONFIG dict, and both EasyOCR readers.
    Output: Extracted OCR text string (empty string if an error occurs).
    """
    pdf_dir = config["train_pdf_dir"] if row["split"] == "train" else config["test_pdf_dir"]
    pdf_path = pdf_dir / f"{row['file_id']}.pdf"
    reader = ja_reader if row["language"] == "ja" else vi_reader

    # Catch errors on a single broken page so a 1-hour job never crashes at page 700.
    try:
        return extract_ocr_text(reader, pdf_path, row["page_number"], config["ocr_dpi"])
    except Exception as error:
        print(f"Warning: OCR failed on {row['file_id']} page {row['page_number']}: {error}")
        return ""


def run_all_ocr(pages_table, config, ja_reader, vi_reader):
    """
    Input: pages_table DataFrame, CONFIG dict, and both EasyOCR readers.
    Output: Updated pages_table with OCR text filled in and saved to Drive.
    """
    pending_mask = (pages_table["needs_ocr"] == True) & (pages_table["ocr_done"] == False)
    pending_indices = list(pages_table[pending_mask].index)
    total_pending = len(pending_indices)
    print("Pages left to OCR:", total_pending)

    start_time = time.time()
    for step, row_index in enumerate(pending_indices, start=1):
        row = pages_table.loc[row_index]
        ocr_text = ocr_single_row(row, config, ja_reader, vi_reader)

        # Only replace existing text if OCR actually found more characters.
        if len(ocr_text) > len(str(row["text"])):
            pages_table.at[row_index, "text"] = ocr_text
            pages_table.at[row_index, "char_count"] = len(ocr_text)
        pages_table.at[row_index, "ocr_done"] = True

        if step % config["checkpoint_every"] == 0 or step == total_pending:
            pages_table.to_parquet(config["pages_parquet_path"], index=False)
            minutes = (time.time() - start_time) / 60
            print(f"Checkpoint saved: {step}/{total_pending} pages ({minutes:.1f} min elapsed)")

    return pages_table


pages_table = load_or_create_checkpoint(pages_table, CONFIG["pages_parquet_path"])
pages_table = run_all_ocr(pages_table, CONFIG, ja_reader, vi_reader)

# Final check to confirm how much text OCR recovered.
ocr_rows = pages_table[pages_table["needs_ocr"] == True]
print("\n=== PREPROCESSING COMPLETE ===")
print("Total pages saved:", len(pages_table))
print("OCR pages completed:", int(ocr_rows["ocr_done"].sum()), "/", len(ocr_rows))
print("OCR pages now with >= 50 chars:", int((ocr_rows["char_count"] >= 50).sum()))
print("Saved file:", CONFIG["pages_parquet_path"])

Created new checkpoint: /content/drive/MyDrive/doculens_lava/outputs/pages.parquet
Pages left to OCR: 868
Checkpoint saved: 50/868 pages (1.9 min elapsed)
Checkpoint saved: 100/868 pages (3.7 min elapsed)
MuPDF error: format error: No default Layer config

Checkpoint saved: 150/868 pages (6.4 min elapsed)
Checkpoint saved: 200/868 pages (8.3 min elapsed)
Checkpoint saved: 250/868 pages (10.4 min elapsed)
Checkpoint saved: 300/868 pages (13.0 min elapsed)
Checkpoint saved: 350/868 pages (16.3 min elapsed)
Checkpoint saved: 400/868 pages (19.7 min elapsed)
Checkpoint saved: 450/868 pages (21.5 min elapsed)
Checkpoint saved: 500/868 pages (22.7 min elapsed)
Checkpoint saved: 550/868 pages (24.2 min elapsed)
Checkpoint saved: 600/868 pages (26.3 min elapsed)
Checkpoint saved: 650/868 pages (28.9 min elapsed)
Checkpoint saved: 700/868 pages (30.1 min elapsed)
Checkpoint saved: 750/868 pages (31.7 min elapsed)
Checkpoint saved: 800/868 pages (34.0 min elapsed)
Checkpoint saved: 850/868 pages

In [7]:
import shutil

for csv_name in ["train.csv", "test.csv", "sample_submission.csv"]:
    shutil.copy(CONFIG["data_dir"] / csv_name, CONFIG["output_dir"] / csv_name)
print("CSVs saved to Drive.")

CSVs saved to Drive.
